In [ ]:
import networkx as nx
import numpy as np
import functools

In [ ]:
@functools.lru_cache(maxsize=None)  # 没有缓存限制
def nodes_degree(graph) -> dict['Node' : 'degree']:
    
    return dict(graph.degree())

@functools.lru_cache(maxsize=None)  # 没有缓存限制
def shortest_path_from_node(graph, source) -> dict['Node' : 'distance']:
    '''
    计算源节点节点source到其他节点的最短路径，包括源节点本身
    返回值为dict[source node to target node, distance]
    '''
    
    return nx.shortest_path_length(graph, source)

@functools.lru_cache(maxsize=None)  
def number_of_node(graph)-> int:
    return nx.number_of_nodes(graph)

In [ ]:
def spread_propagation_node(graph, i: 'node') -> float:
    # - source_node_shortest_path(graph, i): 返回节点i到其他节点的最短路径（字典形式）
    # - nodes_degree(graph): 返回图中所有节点的度（字典形式）
    # - number_of_node(graph): 返回图中节点的总数

    # 获取节点i到其他节点的最短路径
    shortest_paths_from_i = shortest_path_from_node(graph, i)
    
    # 计算图中所有节点的度和平均度
    degree = nodes_degree(graph)
    number_node = number_of_node(graph)
    average_degree = sum(degree.values()) / number_node
    propagation_damping_factor = 1 / average_degree

    # 获取指定邻域下的节点集合
    layer_1 = set(node for node, distance in shortest_paths_from_i.items() if distance == 1)
    layer_2 = set(node for node, distance in shortest_paths_from_i.items() if distance == 2)
    layer_3 = set(node for node, distance in shortest_paths_from_i.items() if distance == 3)

    # 计算各邻域的得分
    score_1 = len(layer_1) * propagation_damping_factor # 一层邻域所有节点得分
    
    number_path_2 = {}  # dict{node:number of path}节点i到目标节点最短距离为2的路径数量
    for node in layer_2:
        number_path2 = len(layer_1 & set(graph.neighbors(node)))
        number_path_2[node] = number_path2
        
    score_2 = sum(number_path_2.values()) * (propagation_damping_factor ** 2)
    
    number_path_3 = {}
    for node in layer_3:
        latest_node = layer_2 & set(graph.neighbors(node))  # 与节点node最近的节点集
        number_path3 = 0
        for node_i in latest_node:
            number_path3 += number_path_2[node_i]
        number_path_3[node] = number_path3
        
    score_3 =sum(number_path_3.values()) * (propagation_damping_factor ** 3)

    # 计算总传播得分
    sp = score_1 + score_2 + score_3

    return sp